<a href="https://colab.research.google.com/github/Rakesh-562/ScoreStats/blob/main/Random_forest_regressor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
!pip install scikit-learn pandas numpy

import zipfile, os, json
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [97]:
ZIP_PATH = "2024 IPL DATA.zip"
EXTRACT_DIR = "ipl_data"

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Update EXTRACT_DIR to point to the actual extracted folder
EXTRACT_DIR = os.path.join(EXTRACT_DIR, "2024 IPL DATA")

files = [f for f in os.listdir(EXTRACT_DIR) if f.endswith(".json")]
print("Total matches:", len(files))

Total matches: 71


In [99]:
matches = []
innings_data = []

for file in files:
    with open(os.path.join(EXTRACT_DIR, file)) as f:
        data = json.load(f)

    match_id = file.replace(".json", "")
    info = data["info"]

    team1, team2 = info["teams"]
    winner = info.get("outcome", {}).get("winner")

    matches.append({
        "match_id": match_id,
        "team1": team1,
        "team2": team2,
        "winner": winner
    })

    for i, inning in enumerate(data["innings"]):
        team = inning["team"]

        total_runs = 0
        wickets = 0
        balls = 0

        for over in inning["overs"]:
            for d in over["deliveries"]:
                total_runs += d["runs"]["total"]
                balls += 1
                if "wickets" in d:
                    wickets += 1

        innings_data.append({
            "match_id": match_id,
            "innings": i+1,
            "team": team,
            "runs": total_runs,
            "wickets": wickets,
            "balls": balls
        })

matches_df = pd.DataFrame(matches)
innings_df = pd.DataFrame(innings_data)
matches_df

,match_id,team1,team2,winner
0,1426262,Rajasthan Royals,Gujarat Titans,Gujarat Titans
1,1422122,Rajasthan Royals,Lucknow Super Giants,Rajasthan Royals
2,1426299,Rajasthan Royals,Chennai Super Kings,Chennai Super Kings
3,1426265,Punjab Kings,Rajasthan Royals,Rajasthan Royals
4,1426276,Mumbai Indians,Rajasthan Royals,Rajasthan Royals
...,...,...,...,...
66,1426268,Sunrisers Hyderabad,Royal Challengers Bengaluru,Sunrisers Hyderabad
67,1426311,Sunrisers Hyderabad,Rajasthan Royals,Sunrisers Hyderabad
68,1426261,Sunrisers Hyderabad,Punjab Kings,Sunrisers Hyderabad
69,1426307,Punjab Kings,Sunrisers Hyderabad,Sunrisers Hyderabad


In [100]:
team_stats = innings_df.groupby("team").agg({
    "runs": "mean",
    "wickets": "mean",
    "balls": "mean"
}).reset_index()

team_stats.columns = ["team", "avg_runs", "avg_wickets", "avg_balls"]

team_stats["run_rate"] = team_stats["avg_runs"] / (team_stats["avg_balls"] / 6)

team_dict = team_stats.set_index("team").to_dict("index")

In [102]:
X = []
y = []

for _, row in matches_df.iterrows():
    t1 = row["team1"]
    t2 = row["team2"]
    winner = row["winner"]

    if winner is None:
        continue
    if t1 not in team_dict or t2 not in team_dict:
        continue

    f1 = team_dict[t1]
    f2 = team_dict[t2]

    feat = [
        f1["avg_runs"], f1["avg_wickets"], f1["run_rate"],
        f2["avg_runs"], f2["avg_wickets"], f2["run_rate"],
    ]

    X.append(feat)
    y.append(1 if winner == t1 else 0)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

Dataset shape: (71, 6)


In [104]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [105]:
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5333333333333333

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.38      0.46         8
           1       0.50      0.71      0.59         7

    accuracy                           0.53        15
   macro avg       0.55      0.54      0.52        15
weighted avg       0.55      0.53      0.52        15


Confusion Matrix:
[[3 5]
 [2 5]]
